## Преобразование чистых текстовых данных в структурированные (csv-формат)

In [5]:
import os
import pandas as pd
import re
from datetime import date

# Месяцы в родительном падеже
MONTHS = {
    "января": 1, "февраля": 2, "марта": 3, "апреля": 4, "мая": 5, "июня": 6,
    "июля": 7, "августа": 8, "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12,
}
MONTH_RE = r"(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)"
DASH_RE = r"[—–-]"  # эм-, эн-, дефис

# Паттерны: сначала «день месяц — день месяц», затем «день—день месяц», затем одиночная дата
PATTERN_CROSS_MONTH = re.compile(
    rf"^(\d{{1,2}})\s+{MONTH_RE}\s*{DASH_RE}\s*(\d{{1,2}})\s+{MONTH_RE}\b",
    re.IGNORECASE,
)
PATTERN_SAME_MONTH_RANGE = re.compile(
    rf"^(\d{{1,2}})\s*{DASH_RE}\s*(\d{{1,2}})\s+{MONTH_RE}\b",
    re.IGNORECASE,
)
PATTERN_SINGLE_DATE = re.compile(
    rf"^(\d{{1,2}})\s+{MONTH_RE}\b",
    re.IGNORECASE,
)


def _month_num(name: str) -> int:
    return MONTHS[name.lower()]


def parse_events_file(source_path: str, out_path: str):
    # Год из имени файла (например, "2001.txt" → "2001")
    m_year = re.search(r"(\d{4})", os.path.basename(source_path))
    if m_year:
        year = int(m_year.group(1))
    else:
        print(f"⚠️ Не удалось распознать год из названия файла: {source_path}")
        return

    with open(source_path, "r", encoding="utf-8") as f:
        text = f.read()

    # По строкам, без пустых
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    data = []
    current_start = None
    current_end = None

    for line in lines:
        # 1) Диапазон с разными месяцами: «25 июля—23 сентября …»
        m = PATTERN_CROSS_MONTH.match(line)
        if m:
            d1, m1, d2, m2 = m.groups()
            y_start = year
            y_end = year
            # если переход через год (например, «декабря—января») — увеличим год конца
            if _month_num(m2) < _month_num(m1) and year:
                # конец в следующем календарном году
                y_end = y_start + 1

            current_start = date(y_start, _month_num(m1), int(d1))
            current_end = date(y_end, _month_num(m2), int(d2))
            event = line[m.end():].lstrip(" \t:—–-—").strip()
            if event:
                data.append((current_start, current_end, event))
            continue

        # 2) Диапазон в одном месяце: «1–30 января …»
        m = PATTERN_SAME_MONTH_RANGE.match(line)
        if m:
            d1, d2, m1 = m.groups()
            current_start = date(int(year), _month_num(m1), int(d1))
            current_end = date(int(year), _month_num(m1), int(d2))
            event = line[m.end():].lstrip(" \t:—–-—").strip()
            if event:
                data.append((current_start, current_end, event))
            continue

        # 3) Одиночная дата: «2 января …» или «2 января:» (без события на той же строке)
        m = PATTERN_SINGLE_DATE.match(line)
        if m:
            d1, m1 = m.groups()
            current_start = date(int(year), _month_num(m1), int(d1))
            current_end = None
            event = line[m.end():].lstrip(" \t:—–-—").strip()
            if event:
                data.append((current_start, current_end or "", event))
            continue

        # 4) Строка-событие под текущей датой/диапазоном
        if current_start:
            data.append((current_start, current_end or "", line))
        else:
            print(f"⚠️ Не удалось распарсить строку: {line}")

    df = pd.DataFrame(data, columns=["date_start", "date_end", "event"])

    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"✅ [{year or '—'}] Экспортировано {len(df)} строк → {out_path}")


def parse_raw_events(folder_path: str, out_path: str):
    """
    Обходит все .txt в папке и парсит каждый.
    """
    files = sorted(f for f in os.listdir(folder_path) if f.lower().endswith(".txt"))

    for name in files:
        source_path = os.path.join(folder_path, name)
        csv_name = os.path.splitext(name)[0] + ".csv"
        csv_path = os.path.join(out_path, csv_name)
        parse_events_file(source_path, csv_path)


def combine_csv_files(folder_path: str):
    """
    Объединяет все CSV-файлы в указанной папке в один DataFrame
    и сохраняет результат в новый CSV-файл.
    Имя файла формата "<первый>-<последний>.csv".
    """
    # Получаем список CSV файлов в папке
    files = sorted([
        f for f in os.listdir(folder_path)
        if f.endswith('.csv') and '-' not in f
    ])
    if not files:
        raise FileNotFoundError("В указанной папке нет CSV-файлов.")

    # Имя итогового файла
    first_file = os.path.splitext(files[0])[0]
    last_file = os.path.splitext(files[-1])[0]
    output_name = f"{first_file}-{last_file}.csv"
    output_path = os.path.join(folder_path, output_name)

    # Читаем и объединяем
    dataframes = []
    for file in files:
        path = os.path.join(folder_path, file)
        try:
            df = pd.read_csv(path)
            dataframes.append(df)
        except Exception as e:
            print(f"⚠️ Ошибка при чтении {file}: {e}")

    combined_df = pd.concat(dataframes, ignore_index=True)
    combined_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    return combined_df


parse_raw_events("../data/events/1_raw", '../data/events/2_struct')
df = combine_csv_files('../data/events/2_struct')
print(f'Размер банка событий: {len(df)}')

✅ [2000] Экспортировано 301 строк → ../data/events/2_struct\2000.csv
✅ [2001] Экспортировано 238 строк → ../data/events/2_struct\2001.csv
✅ [2002] Экспортировано 480 строк → ../data/events/2_struct\2002.csv
✅ [2003] Экспортировано 188 строк → ../data/events/2_struct\2003.csv
✅ [2004] Экспортировано 134 строк → ../data/events/2_struct\2004.csv
✅ [2005] Экспортировано 180 строк → ../data/events/2_struct\2005.csv
✅ [2006] Экспортировано 110 строк → ../data/events/2_struct\2006.csv
✅ [2007] Экспортировано 126 строк → ../data/events/2_struct\2007.csv
✅ [2008] Экспортировано 145 строк → ../data/events/2_struct\2008.csv
✅ [2009] Экспортировано 146 строк → ../data/events/2_struct\2009.csv
✅ [2010] Экспортировано 169 строк → ../data/events/2_struct\2010.csv
✅ [2011] Экспортировано 289 строк → ../data/events/2_struct\2011.csv
✅ [2012] Экспортировано 237 строк → ../data/events/2_struct\2012.csv
✅ [2013] Экспортировано 246 строк → ../data/events/2_struct\2013.csv
✅ [2014] Экспортировано 287 строк 

## Добавление идентификатора uuid v7 в таблицу с событиями

In [6]:
from datetime import datetime
import pandas as pd
import uuid


def add_id_uuid(df: pd.DataFrame, out_path: str) -> pd.DataFrame:
    """
    Добавляет колонку id (UUIDv7) в DataFrame на основе даты события.
    ⚠️ Перед выполнением запрашивает подтверждение в консоли.
    """

    confirm = input("⚠️ Добавить UUIDv7 в DataFrame? Это изменит структуру данных. [y/n]: ").strip().lower()
    if confirm != "y":
        print("Операция отменена пользователем.")
        return df

    df["date_start"] = pd.to_datetime(df["date_start"], errors="coerce")

    if "date_end" in df.columns:
        df["date_end"] = pd.to_datetime(df["date_end"], errors="coerce")
        mid_date = df.apply(
            lambda r: r["date_start"] + (r["date_end"] - r["date_start"]) / 2
            if pd.notnull(r["date_end"]) else r["date_start"],
            axis=1
        )
    else:
        mid_date = df["date_start"]

    def uuid_v7_from_datetime(dt: datetime) -> uuid.UUID:
        # Переводим в миллисекунды с 1970 года
        timestamp_ms = int(dt.timestamp() * 1000)
        # Берём 48 бит времени и 80 бит случайных данных (UUIDv7 формат)
        time_high = (timestamp_ms >> 16) & 0xFFFFFFFFFFFF
        time_low = timestamp_ms & 0xFFFF
        rand_bits = uuid.uuid4().int & ((1 << 80) - 1)
        value = (time_high << (80 + 16)) | (0x7 << (80 + 12)) | (time_low << 80) | rand_bits
        return uuid.UUID(int=value)

    df["id"] = mid_date.apply(lambda d: uuid_v7_from_datetime(d if pd.notnull(d) else datetime.now()))
    cols = ["id"] + [c for c in df.columns if c != "id"]
    df = df[cols]
    df.to_csv(out_path, index=False, encoding='utf-8-sig')

    print("✅ Колонка 'id' добавлена.")
    return df


add_id_uuid(df, '../data/db/events.csv')

✅ Колонка 'id' добавлена.


,id,date_start,date_end,event
0,00dc6acf-fc00-482a-968c-7cce8d4f5438,2000-01-01,NaT,Деноминация белорусского рубля;
1,00dc6acf-fc00-4057-afdb-327b500359f6,2000-01-01,NaT,"В связи с «проблемой-2000», в Иране объявлен н..."
2,00dc6acf-fc00-465f-8ed3-e93302fd57b9,2000-01-01,NaT,"Вступление в силу закона в Великобритании, сог..."
3,00dc6ff6-7800-4b0b-8f50-960bb06325fa,2000-01-02,NaT,крушение украинского сухогруза типа «река-море...
4,00dc751c-7400-42a6-b4e7-ed2bd30dc331,2000-01-03,NaT,Обстрел из гранатомёта территории российского ...
...,...,...,...,...
5640,01995a1e-fc00-411b-ab73-19d71666c774,2025-09-18,NaT,на Камчатке зафиксировано землетрясение магнит...
5641,0199646b-f400-494c-a917-510a0baae173,2025-09-20,NaT,проведение конкурса песни «Интервидение» в Мос...
5642,019973de-f800-46e8-bae1-c0c0d7718103,2025-09-23,NaT,Международный уголовный суд представил подтвер...
5643,01997e2b-7000-4913-b9fc-0c1556f93948,2025-09-25,NaT,Парламент Кыргызстана объявил о самороспуске.


## Наполняем файл с тэгами странами

In [7]:
import json


def update_country_tags(countries_json_path: str, tags_csv_path: str):
    """
    Создаёт или обновляет теги стран в файле тегов (tags.csv) на основе JSON со странами.
    Все создаваемые теги имеют type = "country".
    """

    confirm = input(
        f"⚠️ Обновить файл с тэгами '{tags_csv_path}' на основе '{countries_json_path}'? [y/n]: ").strip().lower()
    if confirm != "y":
        print("Операция отменена пользователем.")
        return

    # === 🔹 Загружаем JSON со странами ===
    with open(countries_json_path, "r", encoding="utf-8") as f:
        countries = json.load(f)

    df_countries = pd.DataFrame(countries)
    df_countries["code"] = df_countries["code3"]
    df_countries["type"] = "country"
    df_countries = df_countries[["code", "name", "type"]]

    # === 🔹 Загружаем существующие теги, если файл есть ===
    if os.path.exists(tags_csv_path):
        df_tags = pd.read_csv(tags_csv_path, encoding="utf-8-sig")
    else:
        df_tags = pd.DataFrame(columns=["code", "name", "type"])

    # === 🔹 Обновляем или добавляем страны ===
    df_tags = df_tags[~(df_tags["type"] == "country")]  # удаляем старые страны
    df_result = pd.concat([df_tags, df_countries], ignore_index=True)

    # === 🔹 Сохраняем результат ===
    df_result.to_csv(tags_csv_path, index=False, encoding="utf-8-sig")

    print(f"✅ Файл '{tags_csv_path}' успешно обновлён. Добавлено/обновлено {len(df_countries)} тэгов стран.")


update_country_tags('../data/country_codes.json', '../data/db/tags.csv')

✅ Файл '../data/db/tags.csv' успешно обновлён. Добавлено/обновлено 249 тэгов стран.


In [8]:
import pandas as pd
import os


def create_or_update_custom_tags(tags_csv_path: str):
    """
    Создаёт или обновляет теги в файле tags.csv.
    """

    confirm = input(f"⚠️ Создать или обновить теги в '{tags_csv_path}'? [y/N]: ").strip().lower()
    if confirm != "y":
        print("Операция отменена пользователем.")
        return

    # === 🔹 Определяем новые теги вручную ===
    tags = [
        {"code": "PLANE_CRASH", "name": "Авиакатастрофа", "type": "topic"},
        {"code": "SANCTIONS", "name": "Санкции", "type": "topic"},
        {"code": "OIL", "name": "Нефть", "type": "asset"},
    ]

    df_new = pd.DataFrame(tags)

    # === 🔹 Загружаем существующие теги ===
    if os.path.exists(tags_csv_path):
        df_tags = pd.read_csv(tags_csv_path, encoding="utf-8-sig")
    else:
        df_tags = pd.DataFrame(columns=["code", "name", "type"])

    # === 🔹 Обновляем существующие теги по коду ===
    df_tags = df_tags.copy()
    for _, row in df_new.iterrows():
        code = row["code"]
        if code in df_tags["code"].values:
            df_tags.loc[df_tags["code"] == code, ["name", "type"]] = row[["name", "type"]]
        else:
            df_tags.loc[len(df_tags)] = row

    # === 🔹 Сохраняем обновлённый файл ===
    df_tags.to_csv(tags_csv_path, index=False, encoding="utf-8-sig")

    print(f"✅ Файл '{tags_csv_path}' успешно обновлён.")
    print(f"Добавлено или обновлено {len(df_new)} тэгов.")


create_or_update_custom_tags(tags_csv_path='../data/db/tags.csv')

✅ Файл '../data/db/tags.csv' успешно обновлён.
Добавлено или обновлено 3 тэгов.
